In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from pymongo import MongoClient

In [ ]:
BASE_URL = "https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/" # Base URL of the practice website

# 'HEADERS' is a dictionary telling the website who we are.
# Many sites reject requests without a 'User-Agent'.
HEADERS = {                                           # Dictionary of HTTP headers to identify our scraper
    "User-Agent": (                                   # 'User-Agent' tells the server what kind of client is requesting
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "  # Pretend to be a modern browser on Windows 10, 64-bit
        "AppleWebKit/537.36 (KHTML, like Gecko) "     # Rendering engine (WebKit), compatible with Gecko (Firefox engine)
        "Chrome/119.0.0.0 Safari/537.36"              # Specific browser version: Google Chrome 119, built on Safari/WebKit
    )
}

In [ ]:
robots_url = f"{BASE_URL}/robots.txt"   # Build full URL to robots.txt (f-string injects BASE_URL)

try:
    resp = requests.get(robots_url, headers=HEADERS, timeout=10)   # Send GET request to robots.txt
    if resp.status_code == 200:                                    # If request was successful (200 OK)
        print("robots.txt content:\n")                             # Print header message
        print(resp.text)                                           # Print the actual robots.txt rules
    else:                                                          # If response is not 200 (maybe 404, 403, etc.)
        print(f"No robots.txt found (status {resp.status_code})")  # Print warning with status code
except requests.RequestException as e:                             # Catch network-related errors
    print(f"Error fetching robots.txt: {e}")      

In [ ]:


def fetch_html(url):
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=15
        )

        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser")
        else:
            print(f"[warn] {url} http {response.status_code}")
            return None

    except requests.RequestException as e:
        print(f"[error] Request failed for {url}: {e}")
        return None



In [ ]:
htmlsoup = fetch_html(BASE_URL)
htmlsoup

In [ ]:
def parse_page(soup):
    all_data = []  
    
    for a in soup.select("div.az_list_indivisual a"):
        name = a.text.strip()
        link = a.get("href")

        single_soup = fetch_html(link)
        if single_soup is None:
            continue

        disease_data = {
            "name": name,
            "sections": []   
        }

        for h2 in single_soup.select("h2.wp-block-heading"):
            h2_name = h2.get_text(strip=True)

            section_paragraphs = []
            sibling = h2.find_next_sibling()

            while sibling and sibling.name != "h2":
                if sibling.name == "p":
                    section_paragraphs.append(sibling.get_text(strip=True))
                sibling = sibling.find_next_sibling()

            disease_data["sections"].append({
                "title": h2_name,
                "paragraphs": section_paragraphs
            })

        all_data.append(disease_data)

    return all_data

In [ ]:
parsed_content=parse_page(htmlsoup)
parsed_content

In [ ]:
len(parsed_content)

In [ ]:


Client=MongoClient("mongodb://localhost:27017/")
db=Client["FINALPROJECT"]
coll=db["TRYDATA"]

In [ ]:
if parsed_content:  
    for disease in parsed_content:
        
        existing = coll.find_one({"name": disease["name"]})
        if not existing:
            coll.insert_one(disease)
        else:
           
            coll.update_one(
                {"name": disease["name"]},
                {"$set": {"sections": disease["sections"]}}
            )

    print(f" Done! Inserted/Updated {len(parsed_content)} diseases into MongoDB.")
else:
    print(" No data to insert.")